# Loan (UCI credit default): the well-behaved one that still lies

*Notebook for section 2. **Config B**: a proper train / validation / test split, standardized features, a small MLP trained by plain gradient descent, judged by accuracy. Every number the chapter quotes is produced here and printed, so nothing is taken on trust.*

## a. Reading the problem

In [1]:
import numpy as np, os

# --- load the loan data (UCI credit-card default): each row is one client ---
DATA = "data/loan_uci350.csv" if os.path.exists("data/loan_uci350.csv") else "../data/loan_uci350.csv"
A = np.loadtxt(DATA, delimiter=",", skiprows=1)   # header row is just column indices 0..23
X, y = A[:, :-1], A[:, -1].astype(int)
n, d = X.shape

print("=== the data ===")
print("clients :", n)
print("features:", d, "(per client)")
defaulters = int((y == 1).sum())
print(f"defaulters     : {defaulters}  ({defaulters/n:.1%})")
print(f"non-defaulters : {n-defaulters}  ({(n-defaulters)/n:.1%})")

# --- read ONE client top to bottom, so we see what a single row is ---
c = X[0]
print("\n=== one client (row 0) ===")
print("  credit limit   :", int(c[0]))
print("  age            :", int(c[4]))
print("  latest bill    :", int(c[11]))
print("  latest payment :", int(c[17]))
print("  ...23 numbers in all...")
print("  defaulted?     :", int(y[0]), "(1 = yes)")

# --- CONFIG B (declared here, used from section 2b on) ---
CFG = dict(train=0.6, val=0.2, test=0.2, width=16, lr=0.3, epochs=300, K=2, seeds=list(range(5)))
print("\n=== CONFIG (B) ===")
print("  split     : train 60% / val 20% / test 20%   (validation reserved for section 5)")
print("  scale     : standardize, fit on the training part only")
print("  model     : MLP  23 -> 16 (ReLU) -> 2 (softmax)")
print("  loss      : cross-entropy   (what training minimises)")
print("  optimizer : plain gradient descent, learning rate 0.3, 300 epochs")
print("  metric    : accuracy        (how we grade it: the fraction it gets right)")

=== the data ===
clients : 30000
features: 23 (per client)
defaulters     : 6636  (22.1%)
non-defaulters : 23364  (77.9%)

=== one client (row 0) ===
  credit limit   : 20000
  age            : 24
  latest bill    : 3913
  latest payment : 0
  ...23 numbers in all...
  defaulted?     : 1 (1 = yes)

=== CONFIG (B) ===
  split     : train 60% / val 20% / test 20%   (validation reserved for section 5)
  scale     : standardize, fit on the training part only
  model     : MLP  23 -> 16 (ReLU) -> 2 (softmax)
  loss      : cross-entropy   (what training minimises)
  optimizer : plain gradient descent, learning rate 0.3, 300 epochs
  metric    : accuracy        (how we grade it: the fraction it gets right)


## b. By the book

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "svg.fonttype": "none", "font.size": 10,
    "axes.edgecolor": "#c9ced6", "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": "#eef1f4", "grid.linewidth": 0.9,
    "axes.labelcolor": "#4a5460", "text.color": "#1f2d3d",
    "xtick.color": "#8a93a0", "ytick.color": "#8a93a0",
})
ACCENT = "#3a6ea5"
FIG = "figures" if os.path.isdir("figures") else "../figures"

# --- the model: the same small network from section 1 ---
def init(dim, width, K, rng):
    return [rng.standard_normal((dim, width))*0.1, np.zeros(width),
            rng.standard_normal((width, K))*0.1, np.zeros(K)]
def forward(Xb, p):
    W1,b1,W2,b2 = p; a = Xb@W1+b1; h = np.maximum(a,0.0); z = h@W2+b2
    return a,h,z
def softmax(z):
    z = z-z.max(1,keepdims=True); e = np.exp(z); return e/e.sum(1,keepdims=True)
def accuracy(p,Xb,yb):
    return float((forward(Xb,p)[2].argmax(1)==yb).mean())
def standardize(Xtr,Xte):
    m,s = Xtr.mean(0), Xtr.std(0)+1e-9
    return (Xtr-m)/s, (Xte-m)/s

WIDTH, LR, EPOCHS, K = CFG["width"], CFG["lr"], CFG["epochs"], CFG["K"]

def split(seed, how="random"):
    ntest = nval = n//5
    if how == "position":
        idx = np.arange(n)
    elif how == "stratified":
        rng = np.random.default_rng(seed); tr=[]; te=[]
        for cls in (0,1):
            ci = rng.permutation(np.where(y==cls)[0])
            nt = int(round(len(ci)*0.2)); nv = int(round(len(ci)*0.2))
            te += list(ci[:nt]); tr += list(ci[nt+nv:])
        return np.array(tr), np.array(te)
    else:
        idx = np.random.default_rng(seed).permutation(n)
    return idx[ntest+nval:], idx[:ntest]   # train 60%, test 20% (val 20% carved out, reserved)

def train(Xtr, ytr, rng, curve=False):
    p = init(Xtr.shape[1], WIDTH, K, rng); m = len(ytr)
    Y = np.zeros((m,K)); Y[np.arange(m), ytr] = 1.0
    hist = []
    for e in range(EPOCHS):
        a,h,z = forward(Xtr,p); dz = (softmax(z)-Y)/m
        W1,b1,W2,b2 = p
        dW2 = h.T@dz; db2 = dz.sum(0); da = (dz@W2.T)*(a>0)
        dW1 = Xtr.T@da; db1 = da.sum(0)
        p = [W1-LR*dW1, b1-LR*db1, W2-LR*dW2, b2-LR*db2]
        if curve and (e % 10 == 0 or e == EPOCHS-1):
            hist.append((e, accuracy(p,Xtr,ytr)))
    return p, hist

def run(seed, how="random", curve=False):
    tr, te = split(seed, how)
    Xtr, Xte = standardize(X[tr], X[te])
    p, hist = train(Xtr, y[tr], np.random.default_rng(seed+1), curve=curve)
    return accuracy(p, Xte, y[te]), hist

# by the book: one honest split
acc0, hist = run(0, "random", curve=True)
print("by the book (seed 0): TEST ACCURACY =", round(acc0, 4))

# every way of cutting the deck
rand = [run(s, "random")[0] for s in range(5)]
strat = run(0, "stratified")[0]
pos = run(0, "position")[0]
print("across cuts -> random seeds 0..4:", [round(v,3) for v in rand],
      "| mean", round(float(np.mean(rand)),3), "sd", round(float(np.std(rand)),3))
print("            -> stratified:", round(strat,3), "| by file order:", round(pos,3))

# the do-nothing baseline: always predict the majority class "no default"
do_nothing = float((y == 0).mean())
print("do-nothing (always 'no default'):", round(do_nothing, 4))
print("our lead over doing nothing     :", round(float(np.mean(rand)) - do_nothing, 4))

# --- chart 1: the learning curve ---
fig, ax = plt.subplots(figsize=(5.4, 2.7))
ax.plot([e for e,_ in hist], [a for _,a in hist], color=ACCENT, lw=2.2)
ax.set_xlabel("epoch"); ax.set_ylabel("training accuracy"); ax.set_ylim(0.72, 0.84)
ax.set_title("It learns: training accuracy climbs, then settles", loc="left", fontsize=11, color="#1f2d3d")
for sp in ("top","right"): ax.spines[sp].set_visible(False)
fig.tight_layout(); fig.savefig(f"{FIG}/loan_learning.svg"); plt.close(fig)

# --- chart 2: stability across every split ---
labels = ["seed 0","seed 1","seed 2","seed 3","seed 4","stratified","by order"]
vals = rand + [strat, pos]
fig, ax = plt.subplots(figsize=(5.4, 2.7))
ax.scatter(range(len(vals)), vals, color=ACCENT, s=55, zorder=3)
ax.set_xticks(range(len(vals))); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("test accuracy"); ax.set_ylim(0.78, 0.84)
ax.set_title("Rock-steady: every way of splitting lands in the same place", loc="left", fontsize=11, color="#1f2d3d")
for sp in ("top","right"): ax.spines[sp].set_visible(False)
fig.tight_layout(); fig.savefig(f"{FIG}/loan_stability.svg"); plt.close(fig)

# --- chart 3: the deflation (our model vs a model that does nothing) ---
fig, ax = plt.subplots(figsize=(4.4, 3.0))
ax.set_axisbelow(True); ax.xaxis.grid(False)
bars = ax.bar(["our model", "always ‘no’"], [acc0, do_nothing], color=[ACCENT, "#b8c0cc"], width=0.5, zorder=3)
ax.set_ylim(0, 1.0); ax.set_ylabel("accuracy")
ax.set_title("Our model vs answering ‘no’ to everyone", loc="left", fontsize=11, color="#1f2d3d")
for b, v in zip(bars, [acc0, do_nothing]):
    ax.text(b.get_x()+b.get_width()/2, v+0.015, f"{v:.3f}", ha="center", fontsize=11, color="#1f2d3d", fontweight="bold")
for sp in ("top","right"): ax.spines[sp].set_visible(False)
fig.tight_layout(); fig.savefig(f"{FIG}/loan_deflate.svg"); plt.close(fig)

# --- chart 4: where 0.779 comes from (class balance as 100 clients) ---
n_def100 = int(round(float((y == 1).mean()) * 100))   # ~22 of every 100 default
fig, ax = plt.subplots(figsize=(4.7, 3.6))
for i in range(100):
    r, c = divmod(i, 10)
    is_def = i >= (100 - n_def100)
    ax.add_patch(plt.Rectangle((c, -r), 0.86, 0.86,
                 facecolor=("#c85a52" if is_def else "#9db8d6"), edgecolor="white", lw=1.3))
ax.set_xlim(-0.3, 10); ax.set_ylim(-11, 1.5); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Out of every 100 clients, about 78 never default", loc="left", fontsize=11, color="#1f2d3d")
ax.add_patch(plt.Rectangle((0, -10.4), 0.62, 0.62, facecolor="#9db8d6", edgecolor="white"))
ax.text(0.85, -10.1, "won't default", fontsize=9, va="center", color="#4a5460")
ax.add_patch(plt.Rectangle((4.0, -10.4), 0.62, 0.62, facecolor="#c85a52", edgecolor="white"))
ax.text(4.85, -10.1, "will default", fontsize=9, va="center", color="#4a5460")
fig.savefig(f"{FIG}/loan_baseline.svg", bbox_inches="tight"); plt.close(fig)

print("saved learning, stability, deflate, baseline charts | default per 100 =", n_def100)

by the book (seed 0): TEST ACCURACY = 0.8195


across cuts -> random seeds 0..4: [0.82, 0.813, 0.82, 0.817, 0.814] | mean 0.817 sd 0.003
            -> stratified: 0.815 | by file order: 0.808
do-nothing (always 'no default'): 0.7788
our lead over doing nothing     : 0.038


saved learning, stability, deflate, baseline charts | default per 100 = 22


## c. What the number was hiding

In [3]:
# take the number apart: on the by-the-book test set (seed 0), what did the model do group by group?
tr, te = split(0, "random")
Xtr, Xte = standardize(X[tr], X[te])
p = train(Xtr, y[tr], np.random.default_rng(1))[0]
yhat = forward(Xte, p)[2].argmax(1); yt = y[te]
TP = int(((yhat==1)&(yt==1)).sum()); FN = int(((yhat==0)&(yt==1)).sum())
TN = int(((yhat==0)&(yt==0)).sum()); FP = int(((yhat==1)&(yt==0)).sum())
print(f"real defaulters: {TP+FN} -> caught {TP}, missed {FN}   (recall {TP/(TP+FN):.3f})")
print(f"real payers    : {TN+FP} -> cleared {TN}, flagged {FP}")
print("accuracy:", round((TP+TN)/len(te),4), "| share of the score that is just cleared payers:", round(TN/len(te),3))

from matplotlib.patches import Patch
fig, ax = plt.subplots(figsize=(6.4, 2.8))
ax.barh([1.0], [TP], color=ACCENT, height=0.62, zorder=3)
ax.barh([1.0], [FN], left=[TP], color="#c85a52", height=0.62, zorder=3)
ax.barh([0.0], [TN], color=ACCENT, height=0.62, zorder=3)
ax.barh([0.0], [FP], left=[TN], color="#c85a52", height=0.62, zorder=3)
ax.set_yticks([0, 1]); ax.set_yticklabels([f"clients who pay ({TN+FP:,})", f"clients who default ({TP+FN:,})"], fontsize=9)
ax.set_xlabel("number of clients"); ax.set_xlim(0, (TN+FP)*1.03)
ax.set_axisbelow(True); ax.xaxis.grid(True); ax.yaxis.grid(False)
ax.text(TP+FN/2, 1.0, f"missed {FN}", ha="center", va="center", color="white", fontsize=10, fontweight="bold")
ax.text(TN/2, 0.0, f"cleared {TN:,}", ha="center", va="center", color="white", fontsize=9)
ax.legend(handles=[Patch(color=ACCENT, label="got it right"), Patch(color="#c85a52", label="got it wrong")],
          loc="lower right", fontsize=8, frameon=False)
ax.set_title("The 82% is almost all the easy group; most defaulters slip through", loc="left", fontsize=10.5, color="#1f2d3d")
for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
fig.tight_layout(); fig.savefig(f"{FIG}/loan_recall.svg"); plt.close(fig)
print("saved loan_recall.svg")

real defaulters: 1353 -> caught 472, missed 881   (recall 0.349)
real payers    : 4647 -> cleared 4445, flagged 202
accuracy: 0.8195 | share of the score that is just cleared payers: 0.741
saved loan_recall.svg


## d. The number we can trust

In [4]:
# the fix: balanced accuracy, scoring each class on its own then averaging the two
def bal_acc(yh, yt):
    rec  = ((yh==1)&(yt==1)).sum() / max((yt==1).sum(), 1)   # defaulters caught
    spec = ((yh==0)&(yt==0)).sum() / max((yt==0).sum(), 1)   # payers cleared
    return float((rec+spec)/2)

accs, bals = [], []
for s in range(5):
    tr, te = split(s, "random")
    Xtr, Xte = standardize(X[tr], X[te])
    p = train(Xtr, y[tr], np.random.default_rng(s+1))[0]
    yh = forward(Xte, p)[2].argmax(1); yt = y[te]
    accs.append(float((yh==yt).mean())); bals.append(bal_acc(yh, yt))

acc_model, bal_model = float(np.mean(accs)), float(np.mean(bals))
acc_nothing, bal_nothing = do_nothing, 0.5   # answering 'no': recall 0, spec 1 -> balanced 0.5
print(f"OUR MODEL     : accuracy {acc_model:.3f}   balanced {bal_model:.3f}")
print(f"answering 'no' : accuracy {acc_nothing:.3f}   balanced {bal_nothing:.3f}")
print(f"gap on accuracy: {acc_model-acc_nothing:.3f}    gap on balanced: {bal_model-bal_nothing:.3f}")

# --- chart: same model, change the ruler, and the difference appears ---
fig, ax = plt.subplots(figsize=(5.6, 3.1))
xpos = np.arange(2); w = 0.36
b1 = ax.bar(xpos-w/2, [acc_model, bal_model], w, color=ACCENT, label="our model", zorder=3)
b2 = ax.bar(xpos+w/2, [acc_nothing, bal_nothing], w, color="#b8c0cc", label="answering ‘no’", zorder=3)
ax.set_xticks(xpos); ax.set_xticklabels(["accuracy", "balanced accuracy"])
ax.set_ylim(0, 1.0); ax.set_ylabel("score")
ax.set_axisbelow(True); ax.xaxis.grid(False)
ax.legend(loc="upper right", fontsize=9, frameon=False)
ax.set_title("Same model, same data. Change the ruler, and the gap appears.", loc="left", fontsize=10, color="#1f2d3d")
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.015, f"{bar.get_height():.2f}", ha="center", fontsize=9, color="#1f2d3d")
for sp in ("top","right"): ax.spines[sp].set_visible(False)
fig.tight_layout(); fig.savefig(f"{FIG}/loan_balanced.svg"); plt.close(fig)
print("saved loan_balanced.svg")

OUR MODEL     : accuracy 0.817   balanced 0.646
answering 'no' : accuracy 0.779   balanced 0.500
gap on accuracy: 0.038    gap on balanced: 0.146
saved loan_balanced.svg
